# Data preparation

Prepare the application data, save the datasets used by visualization and modeling, then build the complete merged dataset.


In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    if (PROJECT_ROOT.parent / "src").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise FileNotFoundError(
            "Run this notebook from the P4 project root or the notebooks folder."
        )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

index_cols = ["TARGET", "SK_ID_CURR", "SK_ID_BUREAU", "SK_ID_PREV", "index", "DAYS_ID_PUBLISH"]

from src.functions import *

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 10)

# csv_to_pkl()

## Application data for visualization


In [2]:
df_visualization = create_application_visualization_dataset()


## Prepare application train and test data



### Load and merge application data


In [3]:
n_rows = None

with open('./data/application_train.pkl', 'rb') as pickle_file:
    train = pickle.load(pickle_file)
    
with open('./data/application_test.pkl', 'rb') as pickle_file:
    test = pickle.load(pickle_file)

### Debug Mode ###
train = train.iloc[:n_rows, :].copy()
test = test.iloc[:n_rows, :].copy()

test["TARGET"] = np.nan
# Merge application_train and application_test
df_application = pd.concat([train, test], keys=['train', 'test']).copy()
df_application.columns = df_application.columns.str.strip()


### Clean application data


In [4]:
#df_application.AMT_INCOME_TOTAL.sort_values()
df_application = df_application.loc[df_application.AMT_INCOME_TOTAL < 30000000].copy() # remove SK_ID_CURR: 114967 (117,000,000)
#df_application.DAYS_LAST_PHONE_CHANGE.value_counts()
df_application["DAYS_LAST_PHONE_CHANGE"] = (df_application["DAYS_LAST_PHONE_CHANGE"].replace(0.0, np.nan)) # missing data


### Engineer application features


In [5]:
# Categorical age
#age_groups = get_age_groups(df_application)
age_groups = [24, 32, 39, 53, 65]

df_application["AGE_GROUP"] = (df_application["DAYS_BIRTH"].apply(lambda x: get_age_group(x, age_groups)).astype(str))

df_application["CREDIT_INCOME_RATIO"] = (df_application["AMT_CREDIT"] / df_application["AMT_INCOME_TOTAL"])
df_application["ANNUITY_INCOME_RATIO"] = (df_application["AMT_ANNUITY"] / df_application["AMT_INCOME_TOTAL"])
df_application["DAYS_EMPLOYED_RATIO"] = (df_application["DAYS_EMPLOYED"] / df_application["DAYS_BIRTH"])

# ORGANIZATION_TYPE 
# Too much category, at this moment it will be drop
df_application["ORGANIZATION_TYPE"] = (df_application["ORGANIZATION_TYPE"].replace(regex={r":? [Tt]ype \d*": ""}))


### Add polynomial features


In [6]:
df_application = df_application.merge(poly_features(df_application), left_on='SK_ID_CURR', right_index=True, how='left')


### Encode categorical features


In [7]:
# Binary encode for categorical features with two categories
#for bin_feature in ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY','EMERGENCYSTATE_MODE']:
#    df_application[bin_feature], uniques = pd.factorize(df_application[bin_feature])

# Drop features based on the result of Cross-validation version of RFE
# df_application = df_application.drop(columns=DROP_APPLICATION_FEATURES)

#df_application = pd.get_dummies(df_application)
df_application, df_cat = one_hot_encoder(df_application, nan_as_category=False)

method = 'std'
#display(df_application.loc[pd.concat([df_application[continous_cols].transform(lambda x: is_outlier(x, method))], axis=1).any(axis=1), continous_cols])
#df_application = df_application[~pd.concat([df_application[continous_cols].transform(lambda x: is_outlier(x, method))], axis=1).any(axis=1)]

continous_cols = [col for col in df_application.select_dtypes(include='float64').columns if col not in index_cols]

### TEST, too much data are removed
#df_application = df_application[~pd.concat([df_application[continous_cols].transform(lambda x: is_outlier(x, method))], axis=1).any(axis=1)]

#print(method, 'done')

#df_application.drop(['ORGANIZATION_TYPE'], axis=1, inplace=True)


### Save prepared application data


In [8]:
with open('./data/application_prepared.pkl', 'wb') as pickle_file:
    pickle.dump(df_application, pickle_file)


## Complete merged dataset


In [9]:
df_train, df_test = get_merged_dataframe(debug=False, load=False)


application train and application test
application dataframe shape: (356254, 246)
bureau and bureau balance
bureau dataframe shape: (305811, 50)
previous application
previous application shape: (338857, 186)
credit card balance
credit card dataframe shape: (103558, 65)
installments payments
installments dataframe shape: (339587, 17)
pos cash balance
pos cash dataframe shape: (337252, 27)
drop features with over 60% of missing values and ID columns
93
train dataframe shape: (356254, 496)
columns are renamed
final merged dataframe shape: (356254, 496)
Initial df memory usage is 2591.81 MB for 496 columns
Final memory usage is 2388.30 MB - decreased by 7.9%
